# Baseline 100% (full-data ceiling) — Kaggle P100

Train YOLOv8n on **100% of the train split** to get the performance ceiling, in the **same environment as the AL runs** (pinned `ultralytics==8.4.51`, single P100). This removes the M4-vs-Kaggle hardware confound for the ceiling line.

Reuses `scripts/03_train_baseline.py` unchanged (seed=13 = seeds.yaml[0], 30 epochs, patience=30 = no early-stop, imgsz=416, batch=64). The only override is `baseline.cache: disk -> False` because `/kaggle/input` is read-only — the same flip the AL notebook applies.

In [ ]:
!pip install -q ultralytics==8.4.51

In [ ]:
# === CONSTANTS — set these to the SAME Kaggle inputs you used for the AL runs ===
# DATA_YAML below is the path your AL runs actually resolved to (see any saved
# results/*/splits/data_*.yaml -> val/test point at this tree). Verify in Add Input.
CODE_DIR  = "/kaggle/input/ttcs-code"                                          # input: albench/ scripts/ configs/
DATA_YAML = "/kaggle/input/datasets/nhichu/ttcs-data-01/export/data.yaml"      # input: export/ (train/val/test)


In [ ]:
import os, sys
from pathlib import Path
import yaml

WORK = "/kaggle/working"
os.chdir(WORK)
sys.path.insert(0, CODE_DIR)

import torch
assert torch.cuda.is_available(), "Bật accelerator GPU trong Settings."
print("GPU:", torch.cuda.get_device_name(0))
assert Path(CODE_DIR, "albench").exists(), f"albench/ không tồn tại: {CODE_DIR}"
assert Path(CODE_DIR, "scripts", "03_train_baseline.py").exists(), "thiếu 03_train_baseline.py"
assert Path(DATA_YAML).exists(), f"DATA_YAML không tồn tại: {DATA_YAML}"

_d = yaml.safe_load(Path(DATA_YAML).read_text())
_base = Path(_d.get("path", str(Path(DATA_YAML).parent)))
_train = _base / _d["train"]
_n = sum(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in _train.glob("*")) if _train.exists() else 0
assert _n > 0, f"Không có ảnh train dưới {_train}"
print("ảnh train (100%):", _n)

# data.yaml bản đường dẫn đầy đủ (script chỉ resolve train, val/test cần absolute sẵn)
for _k in ("train", "val", "test"):
    _d[_k] = str((_base / _d[_k]).resolve())
_d.pop("path", None)
DATA_YAML_ABS = str(Path(WORK) / "data_abs.yaml")
Path(DATA_YAML_ABS).write_text(yaml.safe_dump(_d))
print("data.yaml (absolute) ->", DATA_YAML_ABS)

cfg = yaml.safe_load(Path(CODE_DIR, "configs", "benchmark.yaml").read_text())
cfg["baseline"]["cache"] = False
CFG_KAGGLE = Path(WORK) / "benchmark_kaggle.yaml"
CFG_KAGGLE.write_text(yaml.safe_dump(cfg))
print("config patched ->", CFG_KAGGLE)

In [ ]:
import importlib.util
_spec = importlib.util.spec_from_file_location(
    "_baseline",
    str(Path(CODE_DIR) / "scripts" / "03_train_baseline.py"),
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.run(str(CFG_KAGGLE), str(Path(CODE_DIR) / "configs" / "seeds.yaml"), DATA_YAML_ABS, 1.0)

In [ ]:
# Hiển thị kết quả ceiling
from pathlib import Path
summ = sorted(Path("/kaggle/working/reports/baseline").glob("baseline_frac1.0_*/SUMMARY.md"))
print(summ[-1].read_text() if summ else "(không thấy SUMMARY.md)")
print("Weights:", sorted(Path('/kaggle/working/runs/baseline').glob('baseline_frac1.0_*/weights/last.pt')))


## Sau khi chạy

1. **Save Version** để giữ output.
2. Tải về `runs/baseline/baseline_frac1.0_seed13/` (weights + results.csv) và `reports/baseline/baseline_frac1.0_seed13/` (SUMMARY.md + convergence_curve.png).
3. Con số cần lấy: **Test mAP50** trong SUMMARY.md — đây là đường ceiling P100 để vẽ chung trục với 4 đường cong AL (cùng máy, hết confound phần cứng).
4. So sánh với baseline M4 cũ (test mAP50 = 0.9371) để định lượng độ lệch do máy.